# 18 - Live API
Run the setup cell first, then the work cells in order. Use the WSL mlop-pj kernel.
These cells change cloud resources when you run them. A pending result needs a wait and a phase retry, not a new job.
Use notebooks 18-22 for the five main parts. Training and log waits can overlap with GitHub setup. See docs/RUN_FINISH.md.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert Path(sys.prefix).name == "mlop-pj", "Select the WSL mlop-pj kernel."
assert "microsoft" in os.uname().release.lower(), "Open this project in WSL."
status = json.loads((ROOT / "reports/azure-blob-upload.json").read_text())
assert status.get("data_uploaded") and status.get("all_hashes_match") and not status["errors"], "Finish 17 first."

def step(phase):
    print("Running:", phase, flush=True)
    process = subprocess.run([sys.executable, "-m", "cloudlayer.finish", phase],
                             cwd=ROOT, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("Phase process failed. Check its stderr before retrying: " + process.stderr)
    result = json.loads(process.stdout)
    print(json.dumps(result, indent=2))
    if result["errors"]:
        raise RuntimeError(f"Check reports/finish-{phase}.json and the matching log in tmp. Stop this group.")
    if result.get("pending"):
        print("Still waiting. Read next in the report; do not recreate the resource or job.")
    return result

print("Root:", ROOT)
print("17 passed. Run this notebook work cells in order.")

Root: /mnt/c/Users/SN23341J001185/Documents/MyWork/ICT/ICTYear4_1/MLop/public_teaching_mlaiops/course/MLOP-PJ
17 passed. Run this notebook work cells in order.


## Package setup
Install the Azure DVC plugin once. Constraints keep the existing model package versions. This installs packages; it does not train or deploy. If installation fails, stop and send the last error.

In [2]:
import importlib.util
if importlib.util.find_spec("dvc_azure") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-cloud.txt",
                    "-c", "requirements-notebook.lock.txt", "-c", "requirements-local.lock.txt"],
                   cwd=ROOT, check=True)
print("Azure DVC plugin ready.")

Azure DVC plugin ready.


## Group 1 — DVC, image, and live API
`assets` pushes the raw-data DVC cache and checks a fresh pull against the raw SHA-256. It tests the code, builds the updated API image, and pushes it to ACR. `deploy` creates the API pull identity and Container App, then checks readiness, golden cases, invalid input, batch input, and warm load.

Inputs: the completed Blob upload, canonical bundle, source files, and pinned image base. Outputs: `finish-assets.json`, `finish-deploy.json`, `azure-smoke.json`, and `azure-load.json` in reports. Look for `assets_ready: true` and `api_ready: true`. Open the returned URL with `/docs` to try the API. The warm p95 target is 300 ms at concurrency 10; `api_ready` alone does not prove that target passed.

This group can take several minutes. If a command times out, check existing resources before retrying.

In [3]:
# Use a fresh process even if the kernel still has older imports.
import json
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())

def fresh_step(phase):
    print("Running in a fresh process:", phase, flush=True)
    process = subprocess.run([sys.executable, "-m", "cloudlayer.finish", phase],
                             cwd=ROOT, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError(process.stderr)
    result = json.loads(process.stdout)
    print(json.dumps(result, indent=2))
    assert not result["errors"], "Check the phase report before retrying."
    return result

assets_file = ROOT / "reports/finish-assets.json"
assets = json.loads(assets_file.read_text()) if assets_file.exists() else {}
if assets.get("assets_ready") and not assets.get("errors"):
    print("Using the completed assets report. No rebuild or push.")
else:
    assets = fresh_step("assets")
assert assets.get("assets_ready"), "Finish the asset checks first."
api = fresh_step("deploy")
assert api.get("api_ready"), "Fix the API checks before continuing."


Using the completed assets report. No rebuild or push.
Running in a fresh process: deploy


{
  "checked_at": "2026-10-11T16:40:24.859167+00:00",
  "phase": "deploy",
  "errors": [],
  "cloud_writes_started": true,
  "url": "https://travel-style-492ea24d.icymushroom-08e3ded0.centralindia.azurecontainerapps.io",
  "image_uri": "acrmloppj492ea24d.azurecr.io/travel-style@sha256:05d1819321f472b39fe1035cb9cee68fc7432a9ad2be4b56afdc8c85af562fcc",
  "app_name": "travel-style-492ea24d",
  "deployment_to_ready_seconds": 24.637418190999597,
  "first_ready_request_ms": 815.5300119979074,
  "first_request_scope": "First successful readiness request after deployment; not an isolated scale-from-zero benchmark.",
  "api_ready": true,
  "smoke_passed": true,
  "canary_passed": true,
  "p95_ms": 813.7695736459136,
  "latency_target_met": false,
  "limits": [
    "Client latency includes internet routing; the 300 ms target remains open if latency_target_met is false.",
    "The staging demo endpoint is public. Do not submit private profiles."
  ]
}


## Cold-start measurement
Run after the warm test. Close `/docs` and stop other requests first. This cell checks whether the app has zero replicas before timing a new request. If replicas do not reach zero yet, the report stays pending. Keep cold latency separate from warm p95.

In [4]:
cold = step("cold")

Running: cold
{
  "checked_at": "2026-10-11T16:41:32.734116+00:00",
  "phase": "cold",
  "errors": [
    "ValueError: No ready revision exists."
  ]
}


RuntimeError: Check reports/finish-cold.json and the matching log in tmp. Stop this group.